# Phân tích BI/DSS cho dữ liệu Olist

Notebook được tổ chức theo một luồng chạy từ trên xuống, không phụ thuộc Google Colab và không khai báo import hoặc hàm giữa bài.

## Thư viện và cấu hình

In [ ]:
# Thư viện chuẩn
from pathlib import Path
import shutil
import warnings

# Thư viện phân tích và trực quan hóa
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.holtwinters import Holt, SimpleExpSmoothing

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1X0smlok_stVNIQL5dB0R4SKthEwDOUNG?usp=drive_link'
EXPECTED_CSV_COUNT = 9
MIN_DELIVERED_ORDERS = 30
ALERT_OTD_THRESHOLD = 0.90
ALERT_MIN_ORDERS = 100


## Hàm dùng chung

In [ ]:
def find_project_root(start=None):
    """Tìm thư mục gốc khi chạy notebook từ root hoặc notebooks/."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, current.parent):
        if (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')


def ensure_dataset(data_dir, drive_url, expected_count=9):
    """Tải dữ liệu khi data/raw chưa có đủ số file CSV yêu cầu."""
    data_dir.mkdir(parents=True, exist_ok=True)
    csv_files = sorted(data_dir.rglob('*.csv'))
    if len(csv_files) < expected_count:
        print(f'Chưa đủ {expected_count} file CSV. Đang tải dữ liệu...')
        gdown.download_folder(
            url=drive_url,
            output=str(data_dir),
            quiet=False,
            use_cookies=False,
        )
        csv_files = sorted(data_dir.rglob('*.csv'))
    if len(csv_files) < expected_count:
        raise FileNotFoundError(
            f'Chỉ tìm thấy {len(csv_files)}/{expected_count} file CSV trong {data_dir}.'
        )
    return {path.name: path for path in csv_files}


def load_olist_tables(csv_paths):
    """Đọc chín bảng nguồn và trả về dictionary thống nhất."""
    filenames = {
        'customers': 'olist_customers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'category_translation': 'product_category_name_translation.csv',
    }
    missing = [name for name in filenames.values() if name not in csv_paths]
    if missing:
        raise FileNotFoundError(f'Thiếu file dữ liệu: {missing}')
    return {name: pd.read_csv(csv_paths[file]) for name, file in filenames.items()}


def seller_metrics(items):
    """Tổng hợp các chỉ số vận hành ở cấp người bán."""
    seller_orders = items.drop_duplicates(['seller_id', 'order_id'])
    delivered = seller_orders[seller_orders['is_delivered']]
    result = pd.DataFrame({
        'orders': seller_orders.groupby('seller_id')['order_id'].nunique(),
        'delivered_orders': delivered.groupby('seller_id')['order_id'].nunique(),
        'canceled_orders': seller_orders[seller_orders['order_status'] == 'canceled'].groupby('seller_id')['order_id'].nunique(),
        'gmv': items.groupby('seller_id')['price'].sum(),
        'gmv_at_risk': items[items['at_risk']].groupby('seller_id')['price'].sum(),
        'review': seller_orders.groupby('seller_id')['review_score'].mean(),
        'otd': 1 - delivered.groupby('seller_id')['is_late'].mean(),
        'avg_delay': delivered[delivered['is_late'] == 1].groupby('seller_id')['delay_days'].mean(),
        'handover': items.groupby('seller_id')['handover_on_time'].mean(),
    })
    fill_columns = ['delivered_orders', 'canceled_orders', 'gmv_at_risk', 'avg_delay']
    result[fill_columns] = result[fill_columns].fillna(0)
    result['cancel_rate'] = result['canceled_orders'] / result['orders']
    return result


def scale_metric(values, lower, upper):
    """Chuẩn hóa chỉ số về thang 0-100; hỗ trợ cả chỉ số thuận và nghịch."""
    return 100 * np.clip((values - lower) / (upper - lower), 0, 1)


def calculate_score(data, weights, parameters):
    return sum(
        weights[name] * scale_metric(data[name], *parameters[name])
        for name in weights
    )


def assign_grade(scores):
    return pd.cut(
        scores,
        [-np.inf, 50, 70, 85, np.inf],
        right=False,
        labels=['D', 'C', 'B', 'A'],
    )


def build_otd_table(delivered_orders, order_base, group_column):
    result = delivered_orders.groupby(group_column).agg(
        don_da_giao=('order_id', 'size'),
        don_tre=('is_late', 'sum'),
    )
    result['otd_%'] = (1 - result['don_tre'] / result['don_da_giao']) * 100
    result['gmv_at_risk'] = (
        order_base[order_base['at_risk']]
        .groupby(group_column)['gmv']
        .sum()
    )
    result['canh_bao'] = (
        (result['otd_%'] < ALERT_OTD_THRESHOLD * 100)
        & (result['don_da_giao'] >= ALERT_MIN_ORDERS)
    )
    return result


def forecast_values(method, train, horizon=3):
    values = train.to_numpy()
    if method == 'Naive':
        return np.repeat(values[-1], horizon)
    if method == 'Trung bình trượt 3 tháng':
        return np.repeat(values[-3:].mean(), horizon)
    if method == 'Hồi quy xu hướng tuyến tính':
        slope, intercept = np.polyfit(np.arange(len(values)), values, 1)
        return intercept + slope * np.arange(len(values), len(values) + horizon)
    if method == 'SES':
        return SimpleExpSmoothing(values, initialization_method='estimated').fit().forecast(horizon)
    if method == 'Holt xu hướng cộng':
        return Holt(values, initialization_method='estimated').fit().forecast(horizon)
    if method == 'Holt xu hướng tắt dần':
        return Holt(values, damped_trend=True, initialization_method='estimated').fit().forecast(horizon)
    raise ValueError(f'Phương pháp dự báo không được hỗ trợ: {method}')


def delay_bucket(days):
    return pd.cut(
        days,
        [-np.inf, 0, 3, 7, 14, np.inf],
        labels=['Đúng hạn', '1-3', '4-7', '8-14', '>14'],
    )


## Nạp dữ liệu

In [ ]:
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'eda_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATHS = ensure_dataset(DATA_DIR, DRIVE_FOLDER_URL, EXPECTED_CSV_COUNT)
tables = load_olist_tables(CSV_PATHS)

customers = tables['customers']
geolocation = tables['geolocation']
order_items = tables['order_items']
payments = tables['payments']
reviews = tables['reviews']
orders = tables['orders']
products = tables['products']
sellers = tables['sellers']
category_translation = tables['category_translation']

table_summary = pd.DataFrame({
    'table': tables.keys(),
    'rows': [len(table) for table in tables.values()],
    'columns': [len(table.columns) for table in tables.values()],
})
print('Project root:', PROJECT_ROOT)
print('Data directory:', DATA_DIR)
display(table_summary)


## Chuẩn bị dữ liệu phân tích

In [ ]:
# Chuyển các cột thời gian của Orders sang datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

# Chuyển các cột thời gian của Reviews sang datetime
reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date'],
    errors='coerce'
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp'],
    errors='coerce'
)

# Chọn review trả lời sau cùng của mỗi đơn
reviews_latest = (
    reviews
    .sort_values(
        ['order_id', 'review_answer_timestamp', 'review_id'],
        na_position='first'
    )
    .drop_duplicates(
        subset='order_id',
        keep='last'
    )
)

# Lấy các đơn đã giao và có đủ ngày giao thực tế, ngày giao dự kiến
delivery = orders[
    (orders['order_status'] == 'delivered') &
    orders['order_delivered_customer_date'].notna() &
    orders['order_estimated_delivery_date'].notna()
].copy()

# Tính số ngày giao trễ
delivery['delay_days_raw'] = (
    delivery['order_delivered_customer_date'].dt.normalize()
    - delivery['order_estimated_delivery_date'].dt.normalize()
).dt.days

# Đơn không trễ được gán delay_days = 0
delivery['delay_days'] = (
    delivery['delay_days_raw']
    .clip(lower=0)
)

# Phân loại tình trạng giao hàng
delivery['delivery_status'] = np.where(
    delivery['delay_days'] > 0,
    'Giao trễ',
    'Đúng hạn'
)

# Ghép Review Score vào dữ liệu giao hàng
delivery_review = delivery.merge(
    reviews_latest[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

# Kiểm tra dữ liệu sau khi xử lý
print('Số đơn đã giao dùng cho phân tích:', len(delivery))
print('Số review mới nhất theo đơn:', len(reviews_latest))
print(
    'Số đơn giao hàng có review:',
    delivery_review['review_score'].notna().sum()
)

# Thời hạn bàn giao cho đơn vị vận chuyển
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# Thông tin giao hàng ở cấp đơn hàng (chỉ đơn đã giao có đủ ngày)
delivery['is_late'] = (delivery['delay_days'] > 0).astype(int)
delivery['lead_time'] = (
    delivery['order_delivered_customer_date'].dt.normalize()
    - delivery['order_purchase_timestamp'].dt.normalize()
).dt.days

# Bảng đơn hàng dùng chung: trạng thái, tháng đặt, review, giao hàng
order_base = (
    orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_carrier_date']]
    .merge(reviews_latest[['order_id', 'review_score']], on='order_id', how='left')
    .merge(delivery[['order_id', 'is_late', 'delay_days', 'lead_time']], on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
)
order_base['is_delivered'] = order_base['is_late'].notna()
order_base['month'] = order_base['order_purchase_timestamp'].dt.to_period('M')

# GMV theo đơn và GMV at Risk: đơn giao trễ hoặc review 1-2 sao
order_gmv = order_items.groupby('order_id')['price'].sum()
order_base['gmv'] = order_base['order_id'].map(order_gmv).fillna(0)
order_base['at_risk'] = (order_base['is_late'] == 1) | (order_base['review_score'] <= 2)

# Bảng dòng sản phẩm có thông tin đơn hàng và trạng thái bàn giao
items = order_items.merge(
    order_base[['order_id', 'order_status', 'is_delivered', 'is_late', 'delay_days',
                'review_score', 'order_delivered_carrier_date', 'at_risk']],
    on='order_id', how='left'
)
items['handover_on_time'] = np.where(
    items['order_delivered_carrier_date'].notna(),
    (items['order_delivered_carrier_date'].dt.normalize()
     <= items['shipping_limit_date'].dt.normalize()).astype(float),
    np.nan
)

print('Số đơn:', order_base['order_id'].nunique())
print('Tổng GMV (BRL):', round(order_items['price'].sum(), 2))
print('Số đơn đã giao:', int(order_base['is_delivered'].sum()))
print('Số dòng sản phẩm đã bàn giao:', int(items['handover_on_time'].notna().sum()))

## Khám phá dữ liệu

## Phân bố trạng thái đơn hàng

In [ ]:
# Tổng hợp số đơn theo trạng thái
status_summary = (
    orders['order_status']
    .value_counts()
    .rename_axis('order_status')
    .reset_index(name='order_count')
)

# Tính tỷ lệ phần trăm
status_summary['percentage'] = (
    status_summary['order_count']
    / status_summary['order_count'].sum()
    * 100
)

# Hiển thị bảng kết quả
display(status_summary)

# Sắp xếp dữ liệu để vẽ biểu đồ
plot_data = status_summary.sort_values(
    'percentage',
    ascending=True
)

# Vẽ biểu đồ
plt.figure(figsize=(9, 5))

bars = plt.barh(
    plot_data['order_status'],
    plot_data['percentage']
)

# Hiển thị tỷ lệ trên từng thanh
for bar, value in zip(
    bars,
    plot_data['percentage']
):
    plt.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        f'{value:.2f}%',
        va='center'
    )

plt.title('Phân bố đơn hàng theo trạng thái')
plt.xlabel('Tỷ lệ đơn hàng (%)')
plt.ylabel('Trạng thái đơn hàng')
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'order_status.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Xu hướng số lượng đơn hàng theo thời gian

In [ ]:
# Tổng hợp số lượng đơn hàng theo tháng
monthly_orders = (
    orders
    .dropna(subset=['order_purchase_timestamp'])
    .set_index('order_purchase_timestamp')
    .resample('MS')['order_id']
    .nunique()
    .reset_index(name='order_count')
)

# Lọc giai đoạn có dữ liệu tương đối đầy đủ
monthly_plot = monthly_orders[
    (monthly_orders['order_purchase_timestamp'] >= '2017-01-01') &
    (monthly_orders['order_purchase_timestamp'] <= '2018-08-01')
].copy()

# Hiển thị bảng kết quả
display(monthly_plot)

# Vẽ biểu đồ xu hướng
plt.figure(figsize=(11, 5))

plt.plot(
    monthly_plot['order_purchase_timestamp'],
    monthly_plot['order_count'],
    marker='o'
)

plt.title('Xu hướng số lượng đơn hàng theo tháng')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng đơn hàng')
plt.xticks(rotation=45)
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'monthly_orders.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Tình trạng giao đúng hạn và giao trễ

In [ ]:
# Tổng hợp số đơn theo tình trạng giao hàng
delivery_summary = (
    delivery['delivery_status']
    .value_counts()
    .rename_axis('delivery_status')
    .reset_index(name='order_count')
)

# Tính tỷ lệ phần trăm
delivery_summary['percentage'] = (
    delivery_summary['order_count']
    / delivery_summary['order_count'].sum()
    * 100
)

# Hiển thị bảng kết quả
display(delivery_summary)

# Vẽ biểu đồ
plt.figure(figsize=(7, 5))

bars = plt.bar(
    delivery_summary['delivery_status'],
    delivery_summary['percentage']
)

# Hiển thị tỷ lệ trên cột
for bar, value in zip(
    bars,
    delivery_summary['percentage']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{value:.2f}%',
        ha='center'
    )

plt.title('Tỷ lệ đơn hàng giao đúng hạn và giao trễ')
plt.xlabel('Tình trạng giao hàng')
plt.ylabel('Tỷ lệ (%)')
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'delivery_status.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Phân bố điểm đánh giá của khách hàng

In [ ]:
# Tổng hợp số review theo từng mức điểm
review_summary = (
    reviews_latest['review_score']
    .value_counts()
    .sort_index()
    .rename_axis('review_score')
    .reset_index(name='review_count')
)

# Tính tỷ lệ phần trăm
review_summary['percentage'] = (
    review_summary['review_count']
    / review_summary['review_count'].sum()
    * 100
)

# Hiển thị bảng kết quả
display(review_summary)

# Vẽ biểu đồ
plt.figure(figsize=(8, 5))

bars = plt.bar(
    review_summary['review_score'].astype(str),
    review_summary['percentage']
)

# Hiển thị tỷ lệ trên cột
for bar, value in zip(
    bars,
    review_summary['percentage']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{value:.2f}%',
        ha='center'
    )

plt.title('Phân bố điểm đánh giá của khách hàng')
plt.xlabel('Điểm đánh giá')
plt.ylabel('Tỷ lệ (%)')
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'review_score.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Mối liên hệ giữa tình trạng giao hàng và điểm đánh giá

In [ ]:
# Lọc các đơn có đầy đủ Review Score
review_delivery_valid = (
    delivery_review
    .dropna(subset=['review_score'])
    .copy()
)

# Đánh dấu review thấp từ 1 đến 2 sao
review_delivery_valid['is_low_review'] = (
    review_delivery_valid['review_score'] <= 2
)

# Tổng hợp review theo tình trạng giao hàng
review_delivery_summary = (
    review_delivery_valid
    .groupby('delivery_status')
    .agg(
        order_count=('order_id', 'nunique'),
        average_review=('review_score', 'mean'),
        low_review_rate=('is_low_review', 'mean')
    )
    .reset_index()
)

# Chuyển tỷ lệ review thấp sang phần trăm
review_delivery_summary['low_review_rate'] *= 100

# Làm tròn kết quả
review_delivery_summary['average_review'] = (
    review_delivery_summary['average_review']
    .round(2)
)

review_delivery_summary['low_review_rate'] = (
    review_delivery_summary['low_review_rate']
    .round(2)
)

# Hiển thị kết quả
display(review_delivery_summary)

In [ ]:
# Chuẩn bị dữ liệu để vẽ biểu đồ
review_avg_plot = review_delivery_summary[
    ['delivery_status', 'average_review']
].copy()

# Vẽ biểu đồ
plt.figure(figsize=(7, 5))

bars = plt.bar(
    review_avg_plot['delivery_status'],
    review_avg_plot['average_review']
)

# Hiển thị giá trị trên cột
for bar, value in zip(
    bars,
    review_avg_plot['average_review']
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f'{value:.2f}',
        ha='center'
    )

plt.title('Điểm đánh giá trung bình theo tình trạng giao hàng')
plt.xlabel('Tình trạng giao hàng')
plt.ylabel('Review Score trung bình')
plt.ylim(0, 5)
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'delivery_review.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Tương quan giữa các thuộc tính ở cấp đơn hàng

In [ ]:
# Tổng hợp Order Items về cấp đơn hàng
order_item_agg = (
    order_items
    .groupby('order_id')
    .agg(
        gmv=('price', 'sum'),
        freight_total=('freight_value', 'sum'),
        item_count=('order_item_id', 'count'),
        seller_count=('seller_id', 'nunique')
    )
    .reset_index()
)

# Tổng hợp Payment về cấp đơn hàng
payment_agg = (
    payments
    .groupby('order_id')
    .agg(
        payment_value=('payment_value', 'sum')
    )
    .reset_index()
)

# Tạo dữ liệu giao hàng ở cấp đơn hàng
delivery_corr = delivery[
    [
        'order_id',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'delay_days'
    ]
].copy()

# Tính thời gian giao hàng
delivery_corr['delivery_lead_time'] = (
    delivery_corr['order_delivered_customer_date'].dt.normalize()
    - delivery_corr['order_purchase_timestamp'].dt.normalize()
).dt.days

# Ghép các bảng về cùng grain order_id
order_corr = (
    orders[['order_id']]
    .merge(
        order_item_agg,
        on='order_id',
        how='left'
    )
    .merge(
        payment_agg,
        on='order_id',
        how='left'
    )
    .merge(
        reviews_latest[
            ['order_id', 'review_score']
        ],
        on='order_id',
        how='left'
    )
    .merge(
        delivery_corr[
            [
                'order_id',
                'delivery_lead_time',
                'delay_days'
            ]
        ],
        on='order_id',
        how='left'
    )
)

# Kiểm tra tránh nhân dòng khi ghép
print('Số dòng:', len(order_corr))
print(
    'Số order_id duy nhất:',
    order_corr['order_id'].nunique()
)

In [ ]:
# Chọn các thuộc tính số
corr_cols = [
    'gmv',
    'freight_total',
    'payment_value',
    'item_count',
    'seller_count',
    'delivery_lead_time',
    'delay_days',
    'review_score'
]

# Tính hệ số tương quan Spearman
corr_matrix = (
    order_corr[corr_cols]
    .corr(method='spearman')
)

# Hiển thị bảng tương quan
display(corr_matrix.round(2))

In [ ]:
# Tên hiển thị của các thuộc tính
corr_labels = [
    'GMV',
    'Freight',
    'Payment',
    'Item count',
    'Seller count',
    'Delivery lead time',
    'Delay days',
    'Review score'
]

# Vẽ heatmap tương quan
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(
    corr_matrix,
    vmin=-1,
    vmax=1,
    cmap='coolwarm'
)

# Thiết lập tên thuộc tính trên hai trục
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))

ax.set_xticklabels(
    corr_labels,
    rotation=45,
    ha='right'
)

ax.set_yticklabels(corr_labels)

# Hiển thị hệ số tương quan trong từng ô
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(
            j,
            i,
            f'{corr_matrix.iloc[i, j]:.2f}',
            ha='center',
            va='center'
        )

plt.colorbar(im)
plt.title(
    'Ma trận tương quan Spearman giữa các thuộc tính ở cấp đơn hàng'
)
plt.tight_layout()

# Lưu và hiển thị biểu đồ
plt.savefig(
    OUTPUT_DIR / 'correlation_heatmap.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Thẻ điểm người bán

In [ ]:
seller_summary = seller_metrics(items)


SCORE_PARAMETERS = {  # tiêu chí: (L, U)
    'review': (3.0, 4.5), 'otd': (0.80, 0.97), 'handover': (0.80, 0.97),
    'avg_delay': (14, 1), 'cancel_rate': (0.02, 0.0)
}
SCORE_WEIGHTS = {'review': 0.30, 'otd': 0.25, 'handover': 0.25, 'avg_delay': 0.10, 'cancel_rate': 0.10}



eligible_sellers = seller_summary[seller_summary['delivered_orders'] >= MIN_DELIVERED_ORDERS].copy()
for k in SCORE_PARAMETERS:
    eligible_sellers['S_' + k] = scale_metric(eligible_sellers[k], *SCORE_PARAMETERS[k])
eligible_sellers['seller_score'] = calculate_score(eligible_sellers, SCORE_WEIGHTS, SCORE_PARAMETERS)
eligible_sellers['grade'] = assign_grade(eligible_sellers['seller_score'])
GRADE_ACTIONS = {'A': 'Ưu tiên', 'B': 'Duy trì', 'C': 'Cảnh báo và hỗ trợ', 'D': 'Đề xuất hạn chế'}
eligible_sellers['action'] = eligible_sellers['grade'].map(GRADE_ACTIONS)

print('Tổng số người bán:', len(seller_summary))
print('Người bán đủ điều kiện (>= 30 đơn đã giao):', len(eligible_sellers))
print('Chưa xếp hạng:', len(seller_summary) - len(eligible_sellers))

In [ ]:
# Kết quả phân hạng người bán
grade_summary = (
    eligible_sellers.groupby('grade', observed=False)
    .agg(so_nguoi_ban=('seller_score', 'size'),
         otd_tb=('otd', 'mean'),
         ban_giao_tb=('handover', 'mean'),
         review_tb=('review', 'mean'),
         gmv=('gmv', 'sum'),
         gmv_at_risk=('gmv_at_risk', 'sum'))
    .sort_index(ascending=False)
)
grade_summary['ty_trong_nguoi_ban_%'] = grade_summary['so_nguoi_ban'] / len(eligible_sellers) * 100
grade_summary['gmv_at_risk_tren_gmv_%'] = grade_summary['gmv_at_risk'] / grade_summary['gmv'] * 100
display(grade_summary.round(3))

In [ ]:
# Mười người bán hạng C - D cần ưu tiên can thiệp, sắp theo GMV at Risk
metric_labels = {'S_review': 'Review', 'S_otd': 'OTD', 'S_handover': 'Bàn giao',
         'S_avg_delay': 'Số ngày trễ', 'S_cancel_rate': 'Tỷ lệ hủy'}
priority_sellers = eligible_sellers[eligible_sellers['grade'].isin(['C', 'D'])].sort_values('gmv_at_risk', ascending=False).head(10).copy()
priority_sellers['tieu_chi_yeu_nhat'] = priority_sellers[list(metric_labels)].apply(
    lambda r: ', '.join(metric_labels[c] for c in r.sort_values().index[:2]), axis=1)
priority_sellers.index = priority_sellers.index.str[:8]
display(priority_sellers[['delivered_orders', 'gmv', 'seller_score', 'grade', 'tieu_chi_yeu_nhat', 'action', 'gmv_at_risk']].round(2))

### Phân tích độ nhạy trọng số

In [ ]:
# Độ nhạy trọng số: tỷ lệ người bán đổi hạng so với trọng số gốc
WEIGHT_SCENARIOS = {
    '25/30/25/10/10': {'review': .25, 'otd': .30, 'handover': .25, 'avg_delay': .10, 'cancel_rate': .10},
    '35/25/20/10/10': {'review': .35, 'otd': .25, 'handover': .20, 'avg_delay': .10, 'cancel_rate': .10},
    '30/25/30/5/10':  {'review': .30, 'otd': .25, 'handover': .30, 'avg_delay': .05, 'cancel_rate': .10},
    '20/20/20/20/20': {'review': .20, 'otd': .20, 'handover': .20, 'avg_delay': .20, 'cancel_rate': .20},
}
sensitivity_results = pd.DataFrame({
    'phuong_an': list(WEIGHT_SCENARIOS),
    'ty_le_doi_hang_%': [
        (assign_grade(calculate_score(eligible_sellers, w, SCORE_PARAMETERS)).astype(str) != eligible_sellers['grade'].astype(str)).mean() * 100
        for w in WEIGHT_SCENARIOS.values()
    ]
})
display(sensitivity_results.round(2))

## Cảnh báo vận hành

Gắn cờ nhóm có OTD dưới 90% và ít nhất 100 đơn đã giao.

In [ ]:
delivered_orders = order_base[order_base['is_delivered']]


month_alert = build_otd_table(delivered_orders, order_base, 'month')
state_alert = build_otd_table(delivered_orders, order_base, 'customer_state')

print('Tháng có OTD dưới 90% (không xét cỡ mẫu):')
display(month_alert[month_alert['otd_%'] < 90].round(2))
print('Tháng bị gắn cờ cảnh báo:')
display(month_alert[month_alert['canh_bao']].round(2))
print('Bang bị gắn cờ cảnh báo, sắp theo GMV at Risk:')
display(state_alert[state_alert['canh_bao']].sort_values('gmv_at_risk', ascending=False).round(2))

## Dự báo số đơn và GMV

Backtest trên ba tháng cuối của chuỗi dữ liệu.

In [ ]:
# Chuỗi tháng 01/2017 - 08/2018
forecast_data = order_base[(order_base['month'] >= '2017-01') & (order_base['month'] <= '2018-08')]
forecast_series = {
    'So don': forecast_data.groupby('month')['order_id'].nunique().astype(float),
    'GMV': items.merge(order_base[['order_id', 'month']], on='order_id')
                .query("month >= '2017-01' and month <= '2018-08'")
                .groupby('month')['price'].sum(),
}


METHODS = ['Naive', 'Trung bình trượt 3 tháng', 'Hồi quy xu hướng tuyến tính',
           'SES', 'Holt xu hướng cộng', 'Holt xu hướng tắt dần']
rows = []
for m in METHODS:
    r = {'phuong_phap': m}
    for name, s in forecast_series.items():
        train, test = s.iloc[:-3], s.iloc[-3:]
        pred = forecast_values(m, train)
        r['MAE ' + name] = np.mean(np.abs(test.values - pred))
        r['MAPE ' + name + ' %'] = np.mean(np.abs(test.values - pred) / test.values) * 100
    rows.append(r)
backtest = pd.DataFrame(rows)
display(backtest.round(2))

In [ ]:
# Dự báo SES trên toàn chuỗi, khoảng dự báo 95% và kịch bản cao điểm tháng 11
forecast_rows = []
for name, s in forecast_series.items():
    fit = SimpleExpSmoothing(s.values, initialization_method='estimated').fit()
    alpha = fit.params['smoothing_level']
    f = fit.forecast(3)
    sd = np.std(s.values - fit.fittedvalues, ddof=1)
    k_peak = s[pd.Period('2017-11', 'M')] / s[[pd.Period(p, 'M') for p in ['2017-08', '2017-09', '2017-10']]].mean()
    forecast_rows.append({
        'chi_tieu': name, 'alpha': alpha,
        'du_bao_moi_thang': f[0], 'can_duoi_95': f[0] - 1.96 * sd, 'can_tren_95': f[0] + 1.96 * sd,
        'K_peak': k_peak, 'kich_ban_cao_diem_11_2018': k_peak * f[2]
    })
forecast_result = pd.DataFrame(forecast_rows)
display(forecast_result.round(3))

## Phân tích What-if

In [ ]:
what_if_orders = order_base[order_base['is_delivered'] & order_base['review_score'].notna()].copy()
R_on = what_if_orders.loc[what_if_orders['is_late'] == 0, 'review_score'].mean()
R_late = what_if_orders.loc[what_if_orders['is_late'] == 1, 'review_score'].mean()
gap = R_on - R_late
R0 = what_if_orders['review_score'].mean()
p0 = what_if_orders['is_late'].mean()
otd0 = 1 - delivered_orders['is_late'].mean()
print(f'R_on = {R_on:.4f}, R_late = {R_late:.4f}, Review Gap = {gap:.4f}, R0 = {R0:.4f}, OTD0 = {otd0:.4%}')

# Kịch bản A: giảm tỷ lệ trễ Delta điểm phần trăm
scen_a = pd.DataFrame({'delta_pp': [0, 1, 2, 3, 4, 5]})
scen_a['otd_kich_ban_%'] = (otd0 + scen_a['delta_pp'] / 100) * 100
scen_a['review_kich_ban'] = R0 + gap * scen_a['delta_pp'] / 100
display(scen_a.round(3))

# Kịch bản B: rút ngắn d ngày; đối chứng theo nhóm trễ

bucket_mean = what_if_orders.groupby(delay_bucket(what_if_orders['delay_days']), observed=False)['review_score'].mean()
rows = []
for d in [0, 1, 2, 3, 5, 7]:
    new_delay = (delivered_orders['delay_days'] - d).clip(lower=0)
    p_all = (new_delay > 0).mean()
    p_rev = ((what_if_orders['delay_days'] - d) > 0).mean()
    r_gap = R0 + gap * (p0 - p_rev)
    r_bucket = delay_bucket((what_if_orders['delay_days'] - d).clip(lower=0)).map(bucket_mean).astype(float).mean()
    rows.append({'d': d, 'otd_kich_ban_%': (1 - p_all) * 100,
                 'lead_time_TB': (delivered_orders['lead_time'] - d).clip(lower=0).mean(),
                 'review_theo_Gap': r_gap, 'review_theo_nhom_tre': r_bucket,
                 'chenh_lech': abs(r_gap - r_bucket)})
scen_b = pd.DataFrame(rows)
display(scen_b.round(4))

## Đối chiếu số liệu nguồn

In [ ]:
source_reconciliation = pd.DataFrame({
    'chi_so': ['Tổng số đơn', 'Số đơn đã giao', 'Số đơn trễ', 'OTD %', 'Review đúng hạn', 'Review trễ', 'GMV (BRL)'],
    'gia_tri': [orders['order_id'].nunique(), int(order_base['is_delivered'].sum()),
                int(delivered_orders['is_late'].sum()), otd0 * 100, R_on, R_late, order_items['price'].sum()]
})
display(source_reconciliation.round(2))

## Xuất kết quả biểu đồ

In [ ]:
# Nén toàn bộ biểu đồ bằng Python để chạy được trên Windows, Linux và Colab
archive_base = PROJECT_ROOT / 'notebooks' / 'olist_analysis_charts'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)
print('Đã tạo file:', archive_path)